In [4]:
import random
import matplotlib.pyplot as plt

print("="*70)
print("FACE RECOGNITION MODEL - TESTING")
print("="*70)

# Verify we have everything from training
print("\nVerifying training variables...")
try:
    print(f"  ✓ model loaded")
    print(f"  ✓ scaler loaded")  
    print(f"  ✓ label_encoder loaded")
    print(f"  ✓ IMAGE_DIR: {IMAGE_DIR}")
    print(f"  ✓ IMAGE_SIZE: {IMAGE_SIZE}")
    print(f"  ✓ Expected features: {scaler.n_features_in_}")
    print(f"  ✓ Recognized people: {', '.join(label_encoder.classes_)}")
except NameError as e:
    print(f"❌ Error: {e}")
    print("Please run the training cell first!")
    raise

# ============================================
# PREDICTION FUNCTION
# ============================================

def predict_person(image_path, show_image=True, verbose=True):
    """
    Predict the person in an image
    
    Args:
        image_path: Path to the image file
        show_image: Whether to display the image with prediction
        verbose: Whether to print detailed results
    
    Returns:
        person_name, confidence, all_probabilities
    """
    # Load image
    img = cv2.imread(image_path)
    
    if img is None:
        print(f"❌ Error: Could not load {image_path}")
        return None, None, None
    
    # Process EXACTLY like training
    img_resized = cv2.resize(img, IMAGE_SIZE)
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    
    # Extract features using training functions
    features = extract_combined_features(img_rgb)
    features = features.reshape(1, -1)
    
    # Scale and predict
    features_scaled = scaler.transform(features)
    prediction = model.predict(features_scaled)[0]
    probabilities = model.predict_proba(features_scaled)[0]
    
    # Get results
    person_name = label_encoder.inverse_transform([prediction])[0]
    confidence = probabilities[prediction]
    
    # Create probability dictionary
    all_probs = {
        label_encoder.inverse_transform([idx])[0]: prob 
        for idx, prob in enumerate(probabilities)
    }
    
    # Display results
    if verbose:
        print("\n" + "="*70)
        print("PREDICTION RESULTS")
        print("="*70)
        print(f"📷 Image: {os.path.basename(image_path)}")
        print(f"👤 Predicted: {person_name}")
        print(f"📊 Confidence: {confidence:.2%}")
        print(f"\nAll Probabilities:")
        for name, prob in sorted(all_probs.items(), key=lambda x: x[1], reverse=True):
            bar = '█' * int(prob * 50)
            print(f"  {name:12s} {prob:6.2%} {bar}")
    
    # Show image
    if show_image:
        plt.figure(figsize=(8, 6))
        plt.imshow(img_rgb)
        plt.axis('off')
        
        # Color based on confidence
        if confidence > 0.8:
            color, status = 'green', 'HIGH CONFIDENCE'
        elif confidence > 0.5:
            color, status = 'orange', 'MEDIUM CONFIDENCE'
        else:
            color, status = 'red', 'LOW CONFIDENCE'
        
        plt.title(
            f'{person_name} | {confidence:.1%} | {status}',
            fontsize=14, fontweight='bold', color=color, pad=20
        )
        plt.tight_layout()
        plt.show()
    
    return person_name, confidence, all_probs

print("\n✓ Prediction function defined")

# ============================================
# QUICK VERIFICATION TEST
# ============================================

print("\n" + "="*70)
print("QUICK VERIFICATION TEST")
print("="*70)

# Test on one image
test_img = os.path.join(IMAGE_DIR, all_images[0])
print(f"\nTesting on: {all_images[0]}")

person, conf, probs = predict_person(test_img, show_image=False, verbose=True)

if person is not None:
    print("\n✓ Prediction function works correctly!")
else:
    print("\n✗ Prediction failed!")

# ============================================
# BATCH ACCURACY TEST
# ============================================

print("\n" + "="*70)
print("BATCH ACCURACY TEST")
print("="*70)

# Get test images (3 per person)
test_images = []
for person in label_encoder.classes_:
    person_images = [
        os.path.join(IMAGE_DIR, f) 
        for f in all_images 
        if f.startswith(person)
    ]
    random.seed(42)
    test_images.extend(random.sample(person_images, min(3, len(person_images))))

print(f"\nTesting {len(test_images)} images...")
print("-"*70)

# Test all images
results = []
correct = 0
total = 0

for img_path in test_images:
    filename = os.path.basename(img_path)
    true_label = filename.split('_')[0]
    
    person, conf, _ = predict_person(img_path, show_image=False, verbose=False)
    
    if person is not None:
        is_correct = person == true_label
        if is_correct:
            correct += 1
        total += 1
        
        status = "✓" if is_correct else "✗"
        conf_emoji = "🟢" if conf > 0.8 else "🟡" if conf > 0.5 else "🔴"
        
        results.append({
            'file': filename,
            'true': true_label,
            'pred': person,
            'conf': conf,
            'correct': is_correct
        })
        
        print(f"{status} {conf_emoji} {filename[:35]:35s} True:{true_label:8s} Pred:{person:8s} {conf:5.1%}")

# ============================================
# SUMMARY STATISTICS
# ============================================

print("\n" + "="*70)
print("📊 TEST SUMMARY")
print("="*70)

accuracy = correct / total if total > 0 else 0
print(f"\n✅ Overall Accuracy: {correct}/{total} = {accuracy:.2%}")

# Per-person accuracy
print(f"\n👥 Per-Person Accuracy:")
for person in label_encoder.classes_:
    person_results = [r for r in results if r['true'] == person]
    if person_results:
        person_correct = sum(1 for r in person_results if r['correct'])
        person_total = len(person_results)
        person_acc = person_correct / person_total * 100
        print(f"   {person:10s}: {person_correct}/{person_total} ({person_acc:.1f}%)")

# Confidence distribution
print(f"\n📈 Confidence Distribution:")
high = sum(1 for r in results if r['conf'] > 0.8)
med = sum(1 for r in results if 0.5 < r['conf'] <= 0.8)
low = sum(1 for r in results if r['conf'] <= 0.5)
print(f"   🟢 High (>80%):     {high:2d} images ({high/len(results)*100:.1f}%)")
print(f"   🟡 Medium (50-80%): {med:2d} images ({med/len(results)*100:.1f}%)")
print(f"   🔴 Low (<50%):      {low:2d} images ({low/len(results)*100:.1f}%)")

# Misclassifications
misclassified = [r for r in results if not r['correct']]
if misclassified:
    print(f"\n⚠️  Misclassified Images ({len(misclassified)}):")
    for r in misclassified:
        print(f"   {r['file']:40s} True:{r['true']:8s} → Pred:{r['pred']:8s} ({r['conf']:.1%})")
else:
    print(f"\n🎉 Perfect Score! All images classified correctly!")

# Average confidence
avg_conf = sum(r['conf'] for r in results) / len(results)
print(f"\n📊 Average Confidence: {avg_conf:.2%}")

# Confidence for correct vs incorrect
correct_confs = [r['conf'] for r in results if r['correct']]
incorrect_confs = [r['conf'] for r in results if not r['correct']]

if correct_confs:
    print(f"   Correct predictions: {sum(correct_confs)/len(correct_confs):.2%} avg confidence")
if incorrect_confs:
    print(f"   Incorrect predictions: {sum(incorrect_confs)/len(incorrect_confs):.2%} avg confidence")

# ============================================
# INTERACTIVE USAGE INSTRUCTIONS
# ============================================

print("\n" + "="*70)
print("💡 INTERACTIVE TESTING")
print("="*70)

print("\n📸 Test individual images with:")
print(f"\n   predict_person(r'{IMAGE_DIR}\\{all_images[0]}')")
print(f"   predict_person(r'{IMAGE_DIR}\\{all_images[1]}', show_image=True)")

print("\n🎯 Function parameters:")
print("   show_image=True   → Display the image")
print("   verbose=True      → Show detailed probabilities")

FACE RECOGNITION MODEL - TESTING

Verifying training variables...
  ✓ model loaded
  ✓ scaler loaded
  ✓ label_encoder loaded
  ✓ IMAGE_DIR: C:\Users\USER\Formative-2-Data-Preprocessing\augmented_images
  ✓ IMAGE_SIZE: (128, 128)
  ✓ Expected features: 26347
  ✓ Recognized people: Belyse, Fidele, Irais, Kerie

✓ Prediction function defined

QUICK VERIFICATION TEST


NameError: name 'all_images' is not defined